In [0]:
CREATE OR REPLACE TABLE mini_project.gold.datacube_kpi AS
WITH gl_monthly AS (
  SELECT 
    company_id,
    company_name,
    department_id,
    department_name,
    year(entry_date) AS year,
    month(entry_date) AS month,
    quarter(entry_date) AS quarter,
    SUM(CASE WHEN pl_section = 'Revenue' THEN credit_amount ELSE 0 END) AS gl_revenue,
    SUM(CASE WHEN pl_section = 'Cost of Sales' THEN ABS(net_amount) ELSE 0 END) AS gl_cogs,
    SUM(CASE WHEN pl_section != 'Revenue' THEN ABS(net_amount) ELSE 0 END) AS gl_total_expenses,
    COUNT(DISTINCT gl_id) AS gl_transaction_count
  FROM mini_project.gold.fact_general_ledger
  GROUP BY 1, 2, 3, 4, 5, 6, 7
),
payroll_monthly AS (
  SELECT 
    company_id,
    department_id,
    year(pay_period_end) AS year,
    month(pay_period_end) AS month,
    COUNT(DISTINCT employee_id) AS employee_count,
    SUM(total_compensation) AS payroll_total_compensation,
    SUM(overtime_pay) AS payroll_overtime,
    SUM(bonus) AS payroll_bonus
  FROM mini_project.gold.fact_payroll
  GROUP BY 1, 2, 3, 4
)
SELECT 
  -- Dimensions
  g.company_id,
  g.company_name,
  g.department_id,
  g.department_name,
  g.year,
  g.month,
  g.quarter,
  
  -- Headcount
  p.employee_count,
  
  -- Revenue & Expenses
  ROUND(g.gl_revenue, 2) AS gl_revenue,
  ROUND(g.gl_cogs, 2) AS gl_cogs,
  ROUND(g.gl_total_expenses, 2) AS gl_total_expenses,
  g.gl_transaction_count,
  
  -- Payroll
  ROUND(p.payroll_total_compensation, 2) AS payroll_total_compensation,
  ROUND(p.payroll_overtime, 2) AS payroll_overtime,
  ROUND(p.payroll_bonus, 2) AS payroll_bonus,
  
  -- Profitability KPIs
  ROUND(g.gl_revenue - g.gl_cogs, 2) AS gross_profit,
  ROUND(g.gl_revenue - g.gl_total_expenses, 2) AS net_profit,
  ROUND((g.gl_revenue - g.gl_cogs) / NULLIF(g.gl_revenue, 0) * 100, 2) AS gross_margin_pct,
  ROUND((g.gl_revenue - g.gl_total_expenses) / NULLIF(g.gl_revenue, 0) * 100, 2) AS net_margin_pct,
  
  -- Productivity KPIs
  ROUND(g.gl_revenue / NULLIF(p.employee_count, 0), 2) AS revenue_per_employee,
  ROUND(g.gl_total_expenses / NULLIF(p.employee_count, 0), 2) AS cost_per_employee
  
FROM gl_monthly g
LEFT JOIN payroll_monthly p 
  ON g.company_id = p.company_id 
  AND g.department_id = p.department_id
  AND g.year = p.year
  AND g.month = p.month;

SELECT * FROM mini_project.gold.datacube_kpi;

In [0]:
/* CREATE OR REPLACE TABLE mini_project.gold.datacube_kpi
PARTITIONED BY (year, month)
AS

WITH gl_monthly AS (
  SELECT 
    company_id,
    department_id,
    date_trunc('month', entry_date) AS period_month,
    year(entry_date) AS year,
    month(entry_date) AS month,
    COUNT(gl_id) AS transaction_count,
    SUM(CASE WHEN pl_section = 'Revenue' THEN credit_amount ELSE 0 END) AS revenue,
    SUM(CASE WHEN pl_section = 'Cost of Sales' THEN ABS(net_amount) ELSE 0 END) AS cogs,
    SUM(CASE WHEN pl_section != 'Revenue' THEN ABS(net_amount) ELSE 0 END) AS total_expenses

  FROM mini_project.gold.fact_general_ledger
  GROUP BY company_id, department_id, period_month, year, month
),

payroll_monthly AS (
  SELECT 
    company_id,
    department_id,
    date_trunc('month', pay_date) AS period_month,

    COUNT(DISTINCT employee_id) AS employee_count,
    SUM(total_compensation) AS payroll_total_compensation,
    SUM(net_salary) AS payroll_net_salary,
    SUM(overtime_pay) AS total_overtime,
    SUM(bonus) AS total_bonus

  FROM mini_project.gold.fact_payroll
  GROUP BY company_id, department_id, period_month
)

SELECT 
  g.company_id,
  g.department_id,
  g.period_month,
  g.year,
  g.month,
  g.transaction_count,

  -- finance
  g.revenue,
  g.cogs,
  g.total_expenses,

  -- payroll
  p.employee_count,
  p.payroll_total_compensation,
  p.payroll_net_salary,
  p.total_overtime,
  p.total_bonus,

  -- KPIs
  (g.revenue - g.cogs) AS gross_profit,
  (g.revenue - g.cogs - g.total_expenses - p.payroll_total_compensation) AS net_profit,

  CASE 
    WHEN g.revenue > 0 
    THEN (g.revenue - g.cogs) / g.revenue * 100 
  END AS gross_margin_pct,

  CASE 
    WHEN g.revenue > 0 
    THEN (g.revenue - g.cogs - g.total_expenses - p.payroll_total_compensation) / g.revenue * 100 
  END AS net_margin_pct,

  CASE 
    WHEN g.revenue > 0 
    THEN p.payroll_total_compensation / g.revenue * 100 
  END AS payroll_ratio

FROM gl_monthly g
LEFT JOIN payroll_monthly p
  ON g.company_id = p.company_id
  AND g.department_id = p.department_id
  AND g.period_month = p.period_month; */